In [4]:
# %% [markdown]
# # Final Integration: Land Cover

# %%
# imports
import pandas as pd
import os 
import geopandas as gpd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

# %%
# Load Environment Variables (re-using your setup logic)
def load_environment():
    """Load environment variables and return them as a dictionary."""
    load_dotenv(Path("../utils/.env"))  
    return {
        "CLEANED_DATASET_FOLDER": os.getenv("CLEANEDDATASET_FOLDER"),
        "PREPROCESSED_LANDCOVER_FOLDER": os.getenv("PREPROCESSED_LANDCOVER_FOLDER"),
        "FINAL_DATASET_FOLDER": os.getenv("MergedDatasets") # Assuming 'cleaned_dataset_folder' is where you want the final merged file.
    }

folders = load_environment()
input_dir = folders["CLEANED_DATASET_FOLDER"]
preprocessed_landcover_folder = folders["PREPROCESSED_LANDCOVER_FOLDER"]
final_output_dir = folders["FINAL_DATASET_FOLDER"]

# Define file paths
MERGED_IN_PATH = os.path.join(final_output_dir, "merged_climate_soil_elevation.csv")
LANDCOVER_IN_PATH = os.path.join(preprocessed_landcover_folder, "preprocessed_landcover_grid.parquet")
FINAL_OUT_PATH = os.path.join(final_output_dir, "landcover_ElevSoilClimate.parquet")

# Ensure final output directory exists
Path(final_output_dir).mkdir(parents=True, exist_ok=True)

# %%
# --- 1. Load the two datasets ---

# Load the previously merged climate/soil/elevation data
try:
    # Note: If the previous step saved merged_clean as CSV, we read CSV.
    # The geometry column needs special handling since CSV doesn't store GeoDataFrames directly.
    merged_data = pd.read_csv(MERGED_IN_PATH) 
    print(f"✅ Loaded Climate/Soil/Elevation data: {merged_data.shape}")
except FileNotFoundError:
    print(f"❌ Error: Climate/Soil/Elevation file not found at {MERGED_IN_PATH}.")
    raise

# Load the preprocessed land cover grid data
try:
    landcover_df = pd.read_parquet(LANDCOVER_IN_PATH)
    print(f"✅ Loaded Preprocessed Land Cover data: {landcover_df.shape}")
except FileNotFoundError:
    print(f"❌ Error: Land Cover file not found at {LANDCOVER_IN_PATH}.")
    raise


# %%
# --- 2. Perform the Merge ---

# Key check: Ensure the column names are identical for the merge key
if 'cell_id' in merged_data.columns and 'cell_id' in landcover_df.columns:
    
    # Use a standard INNER merge on 'cell_id' to keep only grid cells present in BOTH datasets
    # (i.e., cells for which we have both climate/soil and land cover features).
    final_merged_df = merged_data.merge(
        landcover_df, 
        on='cell_id', 
        how='inner'
    )
    
    print("\nSuccessfully performed INNER merge.")
    print(f"Final Merged Data Shape: {final_merged_df.shape}")
    print(f"Total columns: {len(final_merged_df.columns)}")
    
else:
    raise ValueError("Merge failed: 'cell_id' column not found in one or both DataFrames.")


# %%
# --- 3. Final Verification and Cleanup ---

# Verify that essential spatial columns are intact (x, y, geometry string)
print("\nFinal Merged Columns (first 15):")
print(final_merged_df.columns.tolist()[:15])

# Check for NaNs
nan_count = final_merged_df.isnull().sum().sum()
print(f"\nTotal NaN values in final dataset: {nan_count}")

# Show the head
print("\nFinal Merged Dataset Head:")
print(final_merged_df.head())


# %%
# --- 4. Save the Final Dataset ---

# We save as Parquet for efficiency and because it can handle the string representation of 'geometry' better than CSV.
try:
    final_merged_df.to_parquet(FINAL_OUT_PATH, index=False)
    print(f"\n✅ Final Merged Dataset saved successfully to:\n{FINAL_OUT_PATH}")
except Exception as e:
    # Fallback to CSV if Parquet issues persist
    CSV_OUT_PATH = FINAL_OUT_PATH.replace(".parquet", ".csv")
    final_merged_df.to_csv(CSV_OUT_PATH, index=False)
    print(f"⚠️ Parquet failed (Error: {e}). Saved as CSV instead to:\n{CSV_OUT_PATH}")

✅ Loaded Climate/Soil/Elevation data: (23322, 39)



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\

AttributeError: _ARRAY_API not found

✅ Loaded Preprocessed Land Cover data: (23322, 9)

Successfully performed INNER merge.
Final Merged Data Shape: (23322, 47)
Total columns: 47

Final Merged Columns (first 15):
['match_type', 'distance_m', 'cell_id', 'geometry', 'month_1', 'month_2', 'month_3', 'month_4', 'month_5', 'month_6', 'month_7', 'month_8', 'month_9', 'month_10', 'month_11']

Total NaN values in final dataset: 152

Final Merged Dataset Head:
  match_type   distance_m  cell_id  \
0    nearest  5730.516569        0   
1    nearest  4333.944079        1   
2    nearest  3326.267859        2   
3    nearest  3329.154720        3   
4    nearest  3332.056800        4   

                                            geometry  \
0  MULTIPOLYGON (((-8.568909 27.268147000000116, ...   
1  MULTIPOLYGON (((-8.568909 27.368147000000118, ...   
2  MULTIPOLYGON (((-8.568909 27.46814700000012, -...   
3  MULTIPOLYGON (((-8.568909 27.56814700000012, -...   
4  MULTIPOLYGON (((-8.568909 27.668147000000122, ...   

               


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\

AttributeError: _ARRAY_API not found


✅ Final Merged Dataset saved successfully to:
../../MergedDatasets\landcover_ElevSoilClimate.parquet


In [ ]:
import pandas as pd
import os
from pathlib import Path
from dotenv import load_dotenv

# --- 1. Define Paths (Re-using environment setup) ---

def load_environment():
    """Load environment variables and return the output path."""
    load_dotenv(Path("../utils/.env"))  
    # Assuming the final output was saved to the CLEANED_DATASET_FOLDER
    return os.getenv("MergedDatasets")

final_output_dir = load_environment()
FINAL_OUT_PATH = os.path.join(final_output_dir, "landcover_ElevSoilClimate.parquet")

# --- 2. Load and View ---

try:
    # Load the final merged Parquet file
    final_df = pd.read_parquet(FINAL_OUT_PATH)
    
    print(f"✅ Successfully loaded the Final Merged Dataset from: {FINAL_OUT_PATH}\n")
    
    print("## 📊 Dataset Overview")
    print(f"Total rows (Grid Cells): **{len(final_df)}**")
    print(f"Total columns (Features): **{len(final_df.columns)}**")
    
    print("-" * 30)
    
    # Display the column names to check the features
    print("## 📝 Column Information (Features)")
    # Print the column list organized by type/source
    
    climate_cols = [c for c in final_df.columns if c.startswith(('prec_', 'tmax_', 'tmin_'))]
    soil_cols = [c for c in final_df.columns if len(c) <= 7 and c.isupper() and c not in ['cell_id']] # Soil codes are often short/uppercase
    landcover_cols = [c for c in final_df.columns if c.startswith('lcc_')]
    
    print(f"* **Identifiers/Spatial ({len(final_df.columns) - len(climate_cols) - len(soil_cols) - len(landcover_cols)}):** cell_id, geometry, x, y, elevation, match_type, distance_m, ...")
    print(f"* **Climate Features ({len(climate_cols)}):** {', '.join(climate_cols[:3])}, ...")
    print(f"* **Soil Features ({len(soil_cols)}):** {', '.join(soil_cols[:3])}, ...")
    print(f"* **Land Cover (One-Hot Encoded) ({len(landcover_cols)}):** {', '.join(landcover_cols[:3])}, ...")

    print("\n" + "-" * 30)
    
    # Display the first few rows
    print("## 🔎 Head of the Final Merged Data:")
    print(final_df.head())
    
    print("\n" + "-" * 30)
    # Display data types
    print("## 📚 Data Types:")
    # final_df.info()

except FileNotFoundError:
    print(f"❌ Error: Final file not found at {FINAL_OUT_PATH}. Please ensure the final merge step ran correctly.")
except Exception as e:
    print(f"❌ An error occurred while reading the Parquet file: {e}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\

AttributeError: _ARRAY_API not found

✅ Successfully loaded the Final Merged Dataset from: ../../MergedDatasets\landcover_ElevSoilClimate.parquet

## 📊 Dataset Overview
Total rows (Grid Cells): **23322**
Total columns (Features): **47**
------------------------------
## 📝 Column Information (Features)
* **Identifiers/Spatial (40):** cell_id, geometry, x, y, elevation, match_type, distance_m, ...
* **Climate Features (0):** , ...
* **Soil Features (0):** , ...
* **Land Cover (One-Hot Encoded) (7):** lcc_Bare lands, lcc_Croplands, lcc_Forests, ...

------------------------------
## 🔎 Head of the Final Merged Data:

------------------------------
## 📚 Data Types:


In [7]:
final_df.head()

,match_type,distance_m,cell_id,geometry,month_1,month_2,month_3,month_4,month_5,month_6,...,ELEC_COND_soil,elevation,area_log,lcc_Bare lands,lcc_Croplands,lcc_Forests,lcc_Grasslands,lcc_Unclassified,lcc_Vegetation,lcc_Water bodies
0,nearest,5730.516569,0,"MULTIPOLYGON (((-8.568909 27.268147000000116, ...","[np.float64(8.5), np.float64(22.0), np.float64...","[np.float64(11.0), np.float64(25.0), np.float6...","[np.float64(13.0), np.float64(27.0), np.float6...","[np.float64(18.0), np.float64(32.0), np.float6...","[np.float64(20.0), np.float64(33.75), np.float...","[np.float64(23.0), np.float64(36.75), np.float...",...,1.0,452.0,26.539949,True,False,False,False,False,False,False
1,nearest,4333.944079,1,"MULTIPOLYGON (((-8.568909 27.368147000000118, ...","[np.float64(8.0), np.float64(22.0), np.float64...","[np.float64(11.0), np.float64(25.0), np.float6...","[np.float64(13.0), np.float64(27.0), np.float6...","[np.float64(18.0), np.float64(32.0), np.float6...","[np.float64(19.25), np.float64(33.25), np.floa...","[np.float64(22.75), np.float64(36.25), np.floa...",...,1.0,460.0,26.539949,True,False,False,False,False,False,False
2,nearest,3326.267859,2,"MULTIPOLYGON (((-8.568909 27.46814700000012, -...","[np.float64(8.5), np.float64(22.0), np.float64...","[np.float64(11.0), np.float64(24.75), np.float...","[np.float64(13.0), np.float64(27.0), np.float6...","[np.float64(17.5), np.float64(31.5), np.float6...","[np.float64(19.0), np.float64(33.0), np.float6...","[np.float64(22.0), np.float64(36.0), np.float6...",...,1.0,459.0,26.539949,True,False,False,False,False,False,False
3,nearest,3329.154720,3,"MULTIPOLYGON (((-8.568909 27.56814700000012, -...","[np.float64(8.0), np.float64(22.0), np.float64...","[np.float64(10.75), np.float64(24.0), np.float...","[np.float64(13.0), np.float64(27.0), np.float6...","[np.float64(17.0), np.float64(31.0), np.float6...","[np.float64(19.0), np.float64(33.0), np.float6...","[np.float64(22.0), np.float64(36.0), np.float6...",...,1.0,465.0,26.539949,True,False,False,False,False,False,False
4,nearest,3332.056800,4,"MULTIPOLYGON (((-8.568909 27.668147000000122, ...","[np.float64(8.0), np.float64(22.0), np.float64...","[np.float64(10.0), np.float64(24.0), np.float6...","[np.float64(12.5), np.float64(27.0), np.float6...","[np.float64(17.0), np.float64(31.0), np.float6...","[np.float64(19.0), np.float64(33.0), np.float6...","[np.float64(22.0), np.float64(36.0), np.float6...",...,1.0,470.0,26.539949,True,False,False,False,False,False,False


In [8]:
final_df.columns

Index(['match_type', 'distance_m', 'cell_id', 'geometry', 'month_1', 'month_2',
       'month_3', 'month_4', 'month_5', 'month_6', 'month_7', 'month_8',
       'month_9', 'month_10', 'month_11', 'month_12', 'x', 'y',
       'HWSD2_SMU_ID_soil', 'COARSE_soil', 'SAND_soil', 'SILT_soil',
       'CLAY_soil', 'BULK_soil', 'REF_BULK_soil', 'ORG_CARBON_soil',
       'PH_WATER_soil', 'TOTAL_N_soil', 'CEC_SOIL_soil', 'CEC_CLAY_soil',
       'CEC_EFF_soil', 'TEB_soil', 'BSAT_soil', 'ALUM_SAT_soil', 'ESP_soil',
       'TCARBON_EQ_soil', 'GYPSUM_soil', 'ELEC_COND_soil', 'elevation',
       'area_log', 'lcc_Bare lands', 'lcc_Croplands', 'lcc_Forests',
       'lcc_Grasslands', 'lcc_Unclassified', 'lcc_Vegetation',
       'lcc_Water bodies'],
      dtype='object')

In [9]:
len(final_df)

23322